# Assignment 2: Milestone I Natural Language Processing
## Task 2&3
#### Student Name: XXXX XXXX
#### Student ID: 000000


Environment: Python 3 and Jupyter notebook

Libraries used: please include all the libraries you used in your assignment, e.g.,:
* pandas
* re
* numpy

## Introduction
You should give a brief information of this assessment task here.

<span style="color: red"> Note that this is a sample notebook only. You will need to fill in the proper markdown and code blocks. You might also want to make necessary changes to the structure to meet your own needs. Note also that any generic comments written in this notebook are to be removed and replace with your own words.</span>

## Importing libraries 

In [2]:
# Code to import libraries as you need in this assessment, e.g.,
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
import gensim.downloader as api
import warnings
warnings.filterwarnings('ignore')

## Task 2. Generating Feature Representations for Clothing Items Reviews

...... Sections and code blocks on buidling different document feature represetations


<span style="color: red"> You might have complex notebook structure in this section, please feel free to create your own notebook structure. </span>

In [ ]:
# Code to perform the task...


In [5]:
# 1. Load the processed data and vocabulary
df = pd.read_csv('processed.csv')
df.head()

,product_id,brand_name,review_id,review_title,review_text,author,review_date,review_rating,is_a_buyer,product_title,price,avg_product_rating,product_rating_count,product_tags,product_url
0,781070,Olay,16752142,Worth buying 50g one,work claim differ day olay cleanser result,Ashton Dsouza,23/01/2021 15:17,5.0,True,Olay Ultra Lightweight Moisturiser: Luminous W...,1599,4.1,43,NaN,https://www.nykaa.com/olay-ultra-lightweight-m...
1,781070,Olay,14682550,Best cream to start ur day,doe claim thing smoothen ur soft,Amrit Neelam,07/09/2020 15:30,5.0,True,Olay Ultra Lightweight Moisturiser: Luminous W...,1599,4.1,43,NaN,https://www.nykaa.com/olay-ultra-lightweight-m...
2,781070,Olay,15618995,perfect for summers dry for winters,month combin oili greasi absorb quickli moistu...,Sanchi Gupta,13/11/2020 12:24,4.0,True,Olay Ultra Lightweight Moisturiser: Luminous W...,1599,4.1,43,NaN,https://www.nykaa.com/olay-ultra-lightweight-m...
3,781070,Olay,13474509,Not a moisturizer,oili whip act great base primer smoothen doe m...,Ruchi Shah,14/06/2020 11:56,3.0,True,Olay Ultra Lightweight Moisturiser: Luminous W...,1599,4.1,43,NaN,https://www.nykaa.com/olay-ultra-lightweight-m...
4,781070,Olay,16338982,Average,pleas refresh tri,Sukanya Sarkar,22/12/2020 15:24,2.0,True,Olay Ultra Lightweight Moisturiser: Luminous W...,1599,4.1,43,NaN,https://www.nykaa.com/olay-ultra-lightweight-m...


In [6]:
df.isna().sum()

product_id                  0
brand_name                  0
review_id                   0
review_title                0
review_text              1868
author                      0
review_date                 0
review_rating               1
is_a_buyer                  0
product_title               0
price                       0
avg_product_rating          0
product_rating_count        0
product_tags            47782
product_url                 0
dtype: int64

In [7]:
# Handle any NaN values (happens if a review was completely emptied by our filters in Task 1)
df['review_text'] = df['review_text'].fillna('')
df.isna().sum()

product_id                  0
brand_name                  0
review_id                   0
review_title                0
review_text                 0
author                      0
review_date                 0
review_rating               1
is_a_buyer                  0
product_title               0
price                       0
avg_product_rating          0
product_rating_count        0
product_tags            47782
product_url                 0
dtype: int64

In [8]:
# Load vocabulary into a dictionary for fast lookup
vocab_dict = {}
with open('vocab.txt', 'r', encoding='utf-8') as f:
    for line in f:
        word, idx = line.strip().split(':')
        vocab_dict[word] = int(idx)

In [10]:
# 2. Generate Count Vectors (Bag-of-words)
with open('count_vectors.txt', 'w', encoding='utf-8') as f:
    for idx, row in df.iterrows():
        # Using review_id as the identifier requested by the assignment
        review_id = row['review_id'] 
        
        # Split the text string back into a list of words
        tokens = row['review_text'].split()
        
        # Count frequencies only for words that exist in our vocab
        token_counts = Counter([t for t in tokens if t in vocab_dict])
        
        if token_counts:
            # Sort by the integer index from the vocabulary for neatness
            sorted_counts = sorted([(vocab_dict[k], v) for k, v in token_counts.items()])
            # Format: word_integer_index:word_freq separated by comma
            vector_str = ",".join([f"{word_idx}:{freq}" for word_idx, freq in sorted_counts])
            f.write(f"#{review_id},{vector_str}\n")
        else:
            # Empty vector if no valid words
            f.write(f"#{review_id},\n")

In [ ]:
# 3. Load Word Embeddings & Calculate TF-IDF
# We use GloVe 100-dimensional vectors. It is reliable and relatively lightweight.
word_vectors = api.load("glove-wiki-gigaword-100")
# We restrict the TfidfVectorizer strictly to our Task 1 vocabulary
tfidf = TfidfVectorizer(vocabulary=vocab_dict)
tfidf_matrix = tfidf.fit_transform(df['review_text'])

In [ ]:
# 4. Generate Unweighted & Weighted Vectors
unweighted_file = open('unweighted_vectors.txt', 'w', encoding='utf-8')
weighted_file = open('weighted_vectors.txt', 'w', encoding='utf-8')

vector_size = word_vectors.vector_size

for idx, row in df.iterrows():
    review_id = row['review_id']
    tokens = row['review_text'].split()
    
    # Keep only tokens that are in our vocab AND exist in the GloVe model
    valid_tokens = [t for t in tokens if t in vocab_dict and t in word_vectors]
    
    if not valid_tokens:
        # If no valid tokens, output a vector of zeros
        zero_vec_str = ",".join(["0.0"] * vector_size)
        unweighted_file.write(f"#{review_id},{zero_vec_str}\n")
        weighted_file.write(f"#{review_id},{zero_vec_str}\n")
        continue

    # --- Unweighted Embeddings ---
    # Simple mathematical average of all word vectors in the review
    unweighted_vec = np.mean([word_vectors[t] for t in valid_tokens], axis=0)
    unweighted_file.write(f"#{review_id}," + ",".join(map(str, unweighted_vec)) + "\n")
    
    # --- Weighted Embeddings ---
    # Apply TF-IDF weights to the word embeddings
    doc_tfidf = tfidf_matrix[idx]
    weights = []
    vecs = []
    
    for t in valid_tokens:
        vocab_idx = vocab_dict[t]
        weight = doc_tfidf[0, vocab_idx] # Extract the TF-IDF score for this word
        weights.append(weight)
        vecs.append(word_vectors[t])
        
    if sum(weights) == 0:
        weighted_vec = unweighted_vec # Fallback safety measure
    else:
        # Calculate the weighted average using numpy
        weighted_vec = np.average(vecs, axis=0, weights=weights)
        
    weighted_file.write(f"#{review_id}," + ",".join(map(str, weighted_vec)) + "\n")

unweighted_file.close()
weighted_file.close()
print("Task 2 successfully completed! All 3 files are saved and ready.")# 4. Generate Unweighted & Weighted Vectors
unweighted_file = open('unweighted_vectors.txt', 'w', encoding='utf-8')
weighted_file = open('weighted_vectors.txt', 'w', encoding='utf-8')

vector_size = word_vectors.vector_size

for idx, row in df.iterrows():
    review_id = row['review_id']
    tokens = row['review_text'].split()
    
    # Keep only tokens that are in our vocab AND exist in the GloVe model
    valid_tokens = [t for t in tokens if t in vocab_dict and t in word_vectors]
    
    if not valid_tokens:
        # If no valid tokens, output a vector of zeros
        zero_vec_str = ",".join(["0.0"] * vector_size)
        unweighted_file.write(f"#{review_id},{zero_vec_str}\n")
        weighted_file.write(f"#{review_id},{zero_vec_str}\n")
        continue

    # --- Unweighted Embeddings ---
    # Simple mathematical average of all word vectors in the review
    unweighted_vec = np.mean([word_vectors[t] for t in valid_tokens], axis=0)
    unweighted_file.write(f"#{review_id}," + ",".join(map(str, unweighted_vec)) + "\n")
    
    # --- Weighted Embeddings ---
    # Apply TF-IDF weights to the word embeddings
    doc_tfidf = tfidf_matrix[idx]
    weights = []
    vecs = []
    
    for t in valid_tokens:
        vocab_idx = vocab_dict[t]
        weight = doc_tfidf[0, vocab_idx] # Extract the TF-IDF score for this word
        weights.append(weight)
        vecs.append(word_vectors[t])
        
    if sum(weights) == 0:
        weighted_vec = unweighted_vec # Fallback safety measure
    else:
        # Calculate the weighted average using numpy
        weighted_vec = np.average(vecs, axis=0, weights=weights)
        
    weighted_file.write(f"#{review_id}," + ",".join(map(str, weighted_vec)) + "\n")

unweighted_file.close()
weighted_file.close()
print("Task 2 successfully completed! All 3 files are saved and ready.")

### Saving outputs
Save the count vector representation as per spectification.
- count_vectors.txt

In [2]:
# code to save output data...

## Task 3. Clothing Review Classification

...... Sections and code blocks on buidling classification models based on different document feature represetations. 
Detailed comparsions and evaluations on different models to answer each question as per specification. 

<span style="color: red"> You might have complex notebook structure in this section, please feel free to create your own notebook structure. </span>

In [ ]:
# Code to perform the task...


## Summary
Give a short summary and anything you would like to talk about the assessment tasks here.

## Couple of notes for all code blocks in this notebook
- please provide proper comment on your code
- Please re-start and run all cells to make sure codes are runable and include your output in the submission.   
<span style="color: red"> This markdown block can be removed once the task is completed. </span>